# Install Dependecies

In [1]:
%%capture
%pip install -q "transformers>=5,<6"
%pip install matplotlib
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [2]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [3]:
import numpy as np
import pandas as pd
import random
import torch
from datasets import load_dataset
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from tqdm.std import tqdm
from transformers import AutoTokenizer
from transformers import AutoModel
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier

# Import datasets

In [4]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing
Each group member runs the experiment on one language.

In [ ]:
LANGUAGES = "ko"

### 1. Questions
Only the questions are used.

In [7]:
# train
train_questions = df_train.loc[df_train["lang"] == LANGUAGES, "question"].drop_duplicates().tolist()
train_question_set = set(train_questions)

In [9]:
# val
val_questions = df_val.loc[df_val["lang"] == LANGUAGES, "question"].drop_duplicates().tolist()

### 2. Tokenisation

In [11]:
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [12]:
BOS = "<s>"
EOS = "</s>"
UNK = "<UNK>"

In [14]:
# train
train_tokens = [tokenizer.tokenize(question) for question in train_questions]
# val
val_tokens = [tokenizer.tokenize(question) for question in val_questions]

print("train pieces:", sum(len(tokens) for tokens in train_tokens))
print("val pieces:", sum(len(tokens) for tokens in val_tokens))
print("XLM-R <unk> pieces in val:", sum(tokens.count(tokenizer.unk_token) for tokens in val_tokens))

train pieces: 29090
val pieces: 3520
XLM-R <unk> pieces in val: 0


### 3. Unknown-token strategy
The language model only knows the pieces it saw in the training questions. Pieces that occur fewer than `MIN_COUNT` times in training are replaced by `<UNK>`, so the model learns a probability for rare pieces. Any validation piece that is not in the vocabulary is also mapped to `<UNK>`.

We report the `<UNK>` rate, because a high rate makes perplexity look better than it is: all rare pieces share one frequent `<UNK>` token.